# MEG Sensor Space Analysis - Example Usage

This notebook demonstrates how to use the refactored MEG analysis pipeline for sensor space analysis.

## Contents
1. Single Subject Analysis
2. Batch Processing Multiple Subjects
3. Group-Level Statistics
4. Visual Inspection
5. Custom Analysis with Core Functions

**Author:** 2026  
**Last Updated:** March 5, 2026

## Setup and Imports

First, let's import all necessary modules and set up our environment.

In [1]:
import os
import yaml
import numpy as np
import mne

# Import our custom analysis functions
from sensor_space_individual_analysis import run_individual_analysis
from sensor_space_group_statistics import run_group_statistics
from visual_inspection import VisualInspector
from STWM_functions_core import (
    load_preprocessed_data,
    create_epochs,
    compute_time_frequency,
    apply_fooof_multi_channel
)

print("✓ All modules imported successfully")
print(f"MNE version: {mne.__version__}")

/Users/administrator/__PC_HSE/Results/Nikita Otstavnov/SpatialTemporalWorkingMemoryMEG-main/visual_inspection.py:18: DeprecationWarning: 
The `fooof` package is being deprecated and replaced by the `specparam` (spectral parameterization) package.
This version of `fooof` (1.1) is fully functional, but will not be further updated.
New projects are recommended to update to using `specparam` (see Changelog for details).
  from fooof.plts.spectra import plot_spectrum, plot_spectra


✓ All modules imported successfully
MNE version: 1.11.0


## Load Configuration

Load the analysis configuration from the YAML file.

In [2]:
# Load configuration
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully!")
print(f"\nData folder: {config['paths']['data_folder']}")
print(f"Subject: {config['subject']['name']}")
print(f"Conditions: {config['conditions']['condition_1']['name']} vs {config['conditions']['condition_2']['name']}")

Configuration loaded successfully!

Data folder: /Users/administrator/__PC_HSE/Results/Yulia K/Data/S1
Subject: S1
Conditions: SiVe vs CoVe


---
## Example 1: Single Subject Analysis

Analyze a single subject with all preprocessing steps.

In [ ]:
def example_single_subject_analysis():
    """
    Example: Analyze a single subject
    """
    print("="*60)
    print("Example 1: Single Subject Analysis")
    print("="*60 + "\n")
    
    # You can modify configuration here if needed
    # config['subject']['name'] = 'S1'
    # config['subject']['file_name'] = '1_test_1_tsss_mc_trans.fif'
    
    # Run analysis
    print("Running individual analysis...")
    run_individual_analysis(config)
    print("\n✓ Analysis complete!")

# Run the analysis
example_single_subject_analysis()

### Expected Output Files

After running the analysis, you should see:
- **Epochs files**: `*_epochs-epo.fif`
- **Time-Frequency files**: `*_power_*-tfr.h5`, `*_itc_*-tfr.h5`
- **FOOOF results**: `*_ped_crop.npy`, `*_aper_crop.npy`
- **CSD matrices**: `*_csd.h5`

---
## Example 2: Batch Processing Multiple Subjects

Process multiple subjects in a loop.

In [ ]:
def example_batch_subjects():
    """
    Example: Process multiple subjects
    """
    print("="*60)
    print("Example 2: Batch Processing Multiple Subjects")
    print("="*60 + "\n")
    
    # Define subjects
    subjects = [
        {'name': 'S1', 'file': '1_test_1_tsss_mc_trans.fif'},
        {'name': 'S2', 'file': '2_test_1_tsss_mc_trans.fif'},
        {'name': 'S3', 'file': '3_test_1_tsss_mc_trans.fif'},
    ]
    
    # Process each subject
    for subj in subjects:
        print(f"\nProcessing {subj['name']}...")
        config['subject']['name'] = subj['name']
        config['subject']['file_name'] = subj['file']
        
        try:
            run_individual_analysis(config)
            print(f"✓ {subj['name']} completed")
        except Exception as e:
            print(f"✗ Error processing {subj['name']}: {e}")
    
    print("\n✓ Batch processing complete!")

# Uncomment to run batch analysis
# example_batch_subjects()

**Note:** Uncomment the last line above to run batch processing. Make sure all data files are available.

---
## Example 3: Group-Level Statistics

Run statistical analysis across multiple subjects.

In [ ]:
def example_group_statistics():
    """
    Example: Run group-level statistics
    """
    print("="*60)
    print("Example 3: Group-Level Statistics")
    print("="*60 + "\n")
    
    # Set number of subjects
    config['statistics']['num_subjects'] = 3
    
    # Run group analysis
    print("Running group statistics...")
    T_obs, T_obs_plot, clusters, cluster_p_values = run_group_statistics(config)
    
    # Display results
    print("\n" + "="*60)
    print("Results Summary")
    print("="*60)
    print(f"T-values shape: {T_obs.shape}")
    print(f"Number of clusters: {len(clusters)}")
    print(f"Significant clusters (p < 0.05): {np.sum(cluster_p_values < 0.05)}")
    
    if np.any(cluster_p_values < 0.05):
        print("\nSignificant clusters:")
        for i, p in enumerate(cluster_p_values):
            if p < 0.05:
                print(f"  Cluster {i}: p = {p:.4f}")
    
    return T_obs, T_obs_plot, clusters, cluster_p_values

# Uncomment to run group statistics
# T_obs, T_obs_plot, clusters, cluster_p_values = example_group_statistics()

### Visualize Group Results

In [ ]:
# Uncomment to visualize results after running group statistics
# import matplotlib.pyplot as plt
# 
# fig, ax = plt.subplots(figsize=(10, 6))
# im = ax.imshow(T_obs_plot, aspect='auto', origin='lower', cmap='RdBu_r')
# ax.set_xlabel('Time (samples)')
# ax.set_ylabel('Channels')
# ax.set_title('T-values: Condition 1 vs Condition 2')
# plt.colorbar(im, ax=ax, label='T-value')
# plt.tight_layout()
# plt.show()

---
## Example 4: Visual Inspection

Use the visual inspection module to check data quality and results.

In [ ]:
def example_visual_inspection():
    """
    Example: Visual inspection of results
    """
    print("="*60)
    print("Example 4: Visual Inspection")
    print("="*60 + "\n")
    
    # Create inspector
    inspector = VisualInspector(config)
    
    # Load and inspect data
    print("Loading epochs...")
    epochs_S = inspector.load_and_inspect_epochs('S')
    epochs_T = inspector.load_and_inspect_epochs('T')
    
    print("\nLoading time-frequency data...")
    power_S = inspector.load_and_inspect_power('S')
    power_T = inspector.load_and_inspect_power('T')
    
    print("\nCreating comparison report...")
    inspector.create_comparison_report()
    
    print("\n✓ Visual inspection complete!")
    
    return inspector, epochs_S, epochs_T, power_S, power_T

# Uncomment to run visual inspection
# inspector, epochs_S, epochs_T, power_S, power_T = example_visual_inspection()

---
## Example 5: Custom Analysis with Core Functions

Build your own analysis pipeline using core functions.

### Step 1: Load Data

In [ ]:
# Load data
folder = config['paths']['data_folder']
file_name = config['subject']['file_name']
file_path = os.path.join(folder, file_name)

print("Loading data...")
raw_data = load_preprocessed_data(file_path, meg_type='grad')
print(f"✓ Loaded {len(raw_data.ch_names)} channels")
print(f"  Duration: {raw_data.times[-1]:.2f} seconds")
print(f"  Sampling rate: {raw_data.info['sfreq']} Hz")

### Step 2: Extract Events

In [ ]:
# Extract events
print("\nExtracting events...")
events = mne.find_events(raw_data, stim_channel='STI101')
print(f"✓ Found {len(events)} events")

# Show event types
event_ids = np.unique(events[:, 2])
print(f"  Event IDs: {event_ids}")

### Step 3: Create Epochs

In [ ]:
# Create epochs
print("\nCreating epochs...")
event_id = config['conditions']['condition_1']['event_id']
epochs = create_epochs(
    raw_data, events, event_id=event_id,
    tmin=-4.0, tmax=3.5, picks='grad'
)
print(f"✓ Created {len(epochs)} epochs")
print(f"  Epoch duration: {epochs.times[0]:.2f} to {epochs.times[-1]:.2f} seconds")
print(f"  Number of channels: {len(epochs.ch_names)}")

### Step 4: Compute Time-Frequency Representation

In [ ]:
# Compute time-frequency
print("\nComputing time-frequency...")
freqs = np.logspace(np.log10(4), np.log10(80), num=30)
print(f"  Frequency range: {freqs[0]:.2f} - {freqs[-1]:.2f} Hz")
print(f"  Number of frequencies: {len(freqs)}")

power, itc = compute_time_frequency(
    epochs, freqs, n_cycles=5, decim=20, n_jobs=4
)
print(f"✓ TFR computed: {power.data.shape}")
print(f"  (channels, frequencies, time points)")

### Step 5: Visualize Time-Frequency Results

In [ ]:
# Plot time-frequency representation for one channel
import matplotlib.pyplot as plt

# Select a channel to plot
channel_idx = 0
channel_name = power.ch_names[channel_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot power
power_data = power.data[channel_idx, :, :]
im1 = ax1.imshow(power_data, aspect='auto', origin='lower', 
                 extent=[power.times[0], power.times[-1], freqs[0], freqs[-1]],
                 cmap='RdBu_r')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Frequency (Hz)')
ax1.set_title(f'Power - {channel_name}')
plt.colorbar(im1, ax=ax1, label='Power')

# Plot ITC
itc_data = itc.data[channel_idx, :, :]
im2 = ax2.imshow(itc_data, aspect='auto', origin='lower',
                 extent=[itc.times[0], itc.times[-1], freqs[0], freqs[-1]],
                 cmap='viridis')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Frequency (Hz)')
ax2.set_title(f'ITC - {channel_name}')
plt.colorbar(im2, ax=ax2, label='ITC')

plt.tight_layout()
plt.show()

### Step 6: Apply FOOOF Decomposition

In [ ]:
# Apply FOOOF
print("\nApplying FOOOF decomposition...")
spectrum = np.mean(power.data, axis=2)  # Average over time
periodic, aperiodic = apply_fooof_multi_channel(spectrum, freqs)
print(f"✓ FOOOF completed")
print(f"  Periodic component shape: {periodic.shape}")
print(f"  Aperiodic component shape: {aperiodic.shape}")

### Step 7: Visualize FOOOF Results

In [ ]:
# Plot FOOOF decomposition for one channel
fig, ax = plt.subplots(figsize=(10, 6))

# Original spectrum
ax.plot(freqs, spectrum[channel_idx, :], 'k-', linewidth=2, label='Original', alpha=0.7)

# Aperiodic component
ax.plot(freqs, aperiodic[channel_idx, :], 'r--', linewidth=2, label='Aperiodic (1/f)')

# Periodic component
ax.plot(freqs, periodic[channel_idx, :], 'b-', linewidth=2, label='Periodic (oscillations)')

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power')
ax.set_title(f'FOOOF Decomposition - {channel_name}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Summary

This notebook demonstrated:

✅ **Single subject analysis** - Complete pipeline for one subject  
✅ **Batch processing** - Analyze multiple subjects efficiently  
✅ **Group statistics** - Compare conditions across subjects  
✅ **Visual inspection** - Quality control and result visualization  
✅ **Custom analysis** - Build your own pipeline with core functions  

### Next Steps

1. Modify `config.yaml` for your specific experiment
2. Run analyses on your preprocessed MEG data
3. Explore source space analysis (`source_space_individual_analysis.py`)
4. Try connectivity analysis (`connectivity_individual_analysis.py`)

### Resources

- [MNE-Python Documentation](https://mne.tools/)
- [FOOOF Documentation](https://fooof-tools.github.io/fooof/)
- [Repository README](README.md)

---
## Interactive Exploration

Use the cells below to interactively explore your data.

In [ ]:
# Load your own data here
# raw = mne.io.read_raw_fif('your_file.fif', preload=True)
# events = mne.find_events(raw)
# epochs = mne.Epochs(raw, events, event_id=your_event_id, tmin=-1, tmax=2)

# Your custom analysis code here
pass

In [ ]:
# Additional visualization or analysis
pass